# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Freestyle - whch pages are likely to decline in the next 30 days? (Growth/recovery prediction), and out of those at risk, which should an editor review first? (Refresh/Content Opportunity)

I am not picking one of the four predefined lanes. My question combines two of them honestly, the Growth/Recovery prediction and Refresh / Content Opportunity
This is a freestyle direction: my own label (a future-window outcome), my own windows, and a ranked action queue as the output.


**Why this one?**
The starter data shows both sides of the problem are real and learnable.
54.2% of pages are marked "down", so predicting decline has demand. But 14.6%
are "up" and ~7% are "new", so the "recovers / grows" class exists too, the model is not
predicting a one-sided label.


The warehouse gives 17 months of daily rows, enough to build a true
`prior 90 days of features -> next 30 days of outcome` label instead of the starter's
same-window proxy label. This lane is provisional until the end of Week 4 content. My declared fallback is Lane 2 (Refresh / Content Opportunity Scoring). If the work in weeks 2-3 gets too heavy, I step down to Lane 2 and still deliver a valid capstone.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/AhmedWaleedElFar/flyrank-ml-assignments/main/data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df), "| Clients:", df["client_id"].nunique())

print(df["trend_direction"].value_counts().to_string())
down = (df["trend_direction"] == "down").mean()
up = (df["trend_direction"] == "up").mean()
print(f"\nDeclining share: {down*100:.1f}%  | Rising share: {up*100:.1f}%")

Rows: 30000 | Clients: 32
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Declining share: 54.2%  | Rising share: 14.6%


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The decision I improve:** which of a client's thousands of pages should an editor review, because it is most likely to decline in the next 30 days, and of those at risk,
which are actually worth an editor's time.

**Unit of analysis:** a page (one content item). One row per page per time window.

**Output:** a ranked review queue. Each page gets a predicted risk, a reason code, and a
confidence label.

**Who acts, and what they do:** a content editor opens the queue, inspects the top K pages, and refreshes, rewrites, monitors, or protects them. Limited hours mean they can only act on the top of the list.

**Cost of a wrong call:**
- False positive (flag a page that stays fine): the editor wastes scarce hours on a page
  that did not need them — hours stolen from a page that would have helped.
- False negative (miss a page that declines): a visible page keeps sliding, and the lost
  clicks and impressions compound week after week.

Because a refresh is expensive and reviewers are few, **top-K precision matters more than
recall**: the worst failure is drowning the reviewer in false alarms. A list of 50 with ~37
true positives beats a list of 50 with ~12, and the number of pages a reviewer can touch
is fixed either way.

**Why this is not just "train a model":** the value is the whole funnel. An honest
future-window prediction, a transparent opportunity score, reason codes a human can check,
and validation that proves the ranking beats a simple rule on data the model never saw.
The model only earns its place if that proof holds.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible_declining = df[(df["impressions_90d"] >= 500) & (df["trend_direction"] == "down")]
print(f"Visible & declining pages: {len(visible_declining):,} of {len(df):,} "
      f"({len(visible_declining)/len(df)*100:.1f}%)")
print("A reviewer can act on maybe dozens. Ranking is required, not optional.")

Visible & declining pages: 9,961 of 30,000 (33.2%)
A reviewer can act on maybe dozens. Ranking is required, not optional.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter dataset back this for the next 7 weeks:

1. **54.2% of pages are "declining"** (16,262 of 30,000) — the problem is everywhere, so a
   queue that predicts who is next has real demand.
2. **14.6% are rising** (4,388 "up", plus 2,236 "new") — the recovery/growth side of the
   label is learnable, not a one-sided coin flip.
3. **A learned ranking beats a hand-written rule ~3x on precision@50** (0.74 vs 0.24 from
   the starter pipeline) — evidence that "which page first" is a ranking problem worth
   modeling, not a fixed if-else.

The starter label is a same-window proxy; my capstone will re-earn these numbers with a
true future-window label on the warehouse. But as a first look, they say the lane is worth
seven weeks.

In [13]:
import requests

url = "https://raw.githubusercontent.com/AhmedWaleedElFar/flyrank-ml-assignments/main/outputs/model_results.json"
res = requests.get(url).json()

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"Precision@50: rule {base:.3f} (~{round(base*50)}/50)  vs  random forest {rf:.3f} (~{round(rf*50)}/50)")
print(f"Learned ranking ~= {rf/base:.1f}x the rule.")


JSONDecodeError: Extra data: line 1 column 4 (char 3)

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Declining pages: {(df['trend_direction'] == 'down').sum():,} / {len(df):,} "
      f"({(df['trend_direction'] == 'down').mean()*100:.1f}%)")

up_new = ((df["trend_direction"] == "up") | (df["trend_direction"] == "new")).sum()
print(f"Rising or new pages: {up_new:,} ({up_new/len(df)*100:.1f}%)")

import json
import urllib.request

url = "https://raw.githubusercontent.com/AhmedWaleedElFar/flyrank-ml-assignments/main/outputs/model_results.json"
with urllib.request.urlopen(url) as response:
    res = json.loads(response.read().decode())

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"Precision@50: rule {base:.3f} (~{round(base*50)}/50)  vs  random forest {rf:.3f} (~{round(rf*50)}/50)")
print(f"Learned ranking ~= {rf/base:.1f}x the rule.")

Declining pages: 16,262 / 30,000 (54.2%)
Rising or new pages: 6,624 (22.1%)


HTTPError: HTTP Error 404: Not Found

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.